# Statistical Analysis Shortlist
**Goal**: Reduce 72 results to most promising configurations

## Imports

In [1]:
import sys
import math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import t

### Setup

In [2]:
PRIMARY_METRIC = "f1_macro_mean"   # primary metric
PRIMARY_STD    = "f1_macro_std"
N_COL          = "num_folds"
ALPHA          = 0.05  # 95% CI
TOP_N = 10 
OUT_DIR = Path("./results")
RESULTS_CSV = "pen-based_results.csv"
RERUN_CSV = "shortlist_for_rerun_pen-based.csv"

## Load Results CSV

In [3]:
csv_path = OUT_DIR / RESULTS_CSV

if not csv_path.exists():
    raise SystemExit(f"CSV not found: {csv_path}")

df = pd.read_csv(csv_path)

# csv format check
needed_columns = [PRIMARY_METRIC, PRIMARY_STD, N_COL,
        "dataset","metric","k","vote","retention"]
missing = [c for c in needed_columns if c not in df.columns]
if missing:
    raise SystemExit(f"Missing column(s): {missing}")


## Confidence Interval

In [4]:
def mean_ci(mean: float, sd: float, n: int, alpha: float = 0.05):
    """
    (1-alpha) t-based CI for a mean across n folds:
      mean ± t_{1-alpha/2, n-1} * sd / sqrt(n)
    """
    if n is None or n < 2 or pd.isna(sd):
        return (np.nan, np.nan)
    tcrit = t.ppf(1 - alpha/2.0, df=n-1)
    half = tcrit * (sd / math.sqrt(n))
    return (mean - half, mean + half)

def ci_overlap(a, b) -> bool:
    """True if intervals [a1, a2] and [b1, b2] overlap."""
    a1, a2 = a
    b1, b2 = b
    if any(pd.isna(x) for x in (a1, a2, b1, b2)):
        return False
    return not (a2 < b1 or b2 < a1)

## Keeping all results statistically the same at 95% CI

In [5]:
# compute 95% CIs for the primary metric
cis = df.apply(
    lambda r: mean_ci(r[PRIMARY_METRIC], r[PRIMARY_STD], int(r[N_COL]), ALPHA),
    axis=1
)
df[f"{PRIMARY_METRIC}_ci_lo"], df[f"{PRIMARY_METRIC}_ci_hi"] = zip(*cis)

# find the best by mean of primary metric
best_idx = df[PRIMARY_METRIC].idxmax()
best_row = df.loc[best_idx]
best_ci = (best_row[f"{PRIMARY_METRIC}_ci_lo"], best_row[f"{PRIMARY_METRIC}_ci_hi"])

# keep rows whose CI overlaps best CI
survivors = df[df.apply(
    lambda r: ci_overlap(
        (r[f"{PRIMARY_METRIC}_ci_lo"], r[f"{PRIMARY_METRIC}_ci_hi"]),
        best_ci
    ), axis=1
)].copy()

# sort for viewing: higher mean, then lower std
sort_cols = [PRIMARY_METRIC, PRIMARY_STD]
sort_asc  = [False, True]

survivors = survivors.sort_values(sort_cols, ascending=sort_asc)
survivors

,dataset,metric,k,vote,retention,num_folds,n_train_mean,n_test_mean,fit_time_s_mean,fit_time_s_std,...,f1_macro_std,precision_weighted_mean,precision_weighted_std,recall_weighted_mean,recall_weighted_std,f1_weighted_mean,f1_weighted_std,confusion_matrix_json,f1_macro_mean_ci_lo,f1_macro_mean_ci_hi
40,pen-based,cosine,5,borda,RetentionPolicy.ALWAYS_RETAIN,10,9892.8,1099.2,0.001661,0.000054,...,0.001604,0.994504,0.001635,0.994451,0.001646,0.994450,0.001649,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993319,0.995614
46,pen-based,euclidean,5,borda,RetentionPolicy.NEVER_RETAIN,10,9892.8,1099.2,0.001557,0.000052,...,0.001524,0.994323,0.001554,0.994270,0.001575,0.994268,0.001575,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993187,0.995368
52,pen-based,heom,5,borda,RetentionPolicy.NEVER_RETAIN,10,9892.8,1099.2,0.001666,0.000124,...,0.001524,0.994323,0.001554,0.994270,0.001575,0.994268,0.001575,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993187,0.995368
55,pen-based,euclidean,5,borda,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,9892.8,1099.2,0.001710,0.000145,...,0.001416,0.994322,0.001448,0.994269,0.001467,0.994268,0.001468,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993264,0.995290
61,pen-based,heom,5,borda,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,9892.8,1099.2,0.001752,0.000204,...,0.001416,0.994322,0.001448,0.994269,0.001467,0.994268,0.001468,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993264,0.995290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16,pen-based,heom,5,modified_plurality,RetentionPolicy.NEVER_RETAIN,10,9892.8,1099.2,0.001588,0.000092,...,0.001844,0.992521,0.001856,0.992450,0.001862,0.992453,0.001859,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.991166,0.993803
20,pen-based,euclidean,7,modified_plurality,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,9892.8,1099.2,0.001808,0.000194,...,0.001644,0.992253,0.001642,0.992177,0.001632,0.992179,0.001629,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.991035,0.993387
26,pen-based,heom,7,modified_plurality,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,9892.8,1099.2,0.001718,0.000125,...,0.001644,0.992253,0.001642,0.992177,0.001632,0.992179,0.001629,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.991035,0.993387
29,pen-based,euclidean,7,modified_plurality,RetentionPolicy.DD_RETENTION,10,9892.8,1099.2,0.001707,0.000156,...,0.001644,0.992253,0.001642,0.992177,0.001632,0.992179,0.001629,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.991035,0.993387


In [6]:
lb_col = f"{PRIMARY_METRIC}_ci_lo"

# Sort by strongest conservative performance (higher CI lower bound first)
topN_by_lb = (
    survivors
    .sort_values(lb_col, ascending=False)
    .head(TOP_N)
    .copy()
)

# Save the configs for rerun
minimal_cols = ["dataset","metric","k","vote","retention"]
rerun_df = topN_by_lb[minimal_cols]
rerun_path = OUT_DIR / RERUN_CSV
rerun_df.to_csv(rerun_path, index=False)

topN_by_lb

,dataset,metric,k,vote,retention,num_folds,n_train_mean,n_test_mean,fit_time_s_mean,fit_time_s_std,...,f1_macro_std,precision_weighted_mean,precision_weighted_std,recall_weighted_mean,recall_weighted_std,f1_weighted_mean,f1_weighted_std,confusion_matrix_json,f1_macro_mean_ci_lo,f1_macro_mean_ci_hi
40,pen-based,cosine,5,borda,RetentionPolicy.ALWAYS_RETAIN,10,9892.8,1099.2,0.001661,0.000054,...,0.001604,0.994504,0.001635,0.994451,0.001646,0.994450,0.001649,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993319,0.995614
55,pen-based,euclidean,5,borda,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,9892.8,1099.2,0.001710,0.000145,...,0.001416,0.994322,0.001448,0.994269,0.001467,0.994268,0.001468,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993264,0.995290
61,pen-based,heom,5,borda,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,9892.8,1099.2,0.001752,0.000204,...,0.001416,0.994322,0.001448,0.994269,0.001467,0.994268,0.001468,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993264,0.995290
70,pen-based,heom,5,borda,RetentionPolicy.DD_RETENTION,10,9892.8,1099.2,0.001674,0.000141,...,0.001365,0.994234,0.001399,0.994179,0.001417,0.994177,0.001418,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993210,0.995163
64,pen-based,euclidean,5,borda,RetentionPolicy.DD_RETENTION,10,9892.8,1099.2,0.001690,0.000154,...,0.001365,0.994234,0.001399,0.994179,0.001417,0.994177,0.001418,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993210,0.995163
46,pen-based,euclidean,5,borda,RetentionPolicy.NEVER_RETAIN,10,9892.8,1099.2,0.001557,0.000052,...,0.001524,0.994323,0.001554,0.994270,0.001575,0.994268,0.001575,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993187,0.995368
52,pen-based,heom,5,borda,RetentionPolicy.NEVER_RETAIN,10,9892.8,1099.2,0.001666,0.000124,...,0.001524,0.994323,0.001554,0.994270,0.001575,0.994268,0.001575,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993187,0.995368
49,pen-based,cosine,5,borda,RetentionPolicy.NEVER_RETAIN,10,9892.8,1099.2,0.001699,0.000195,...,0.001441,0.994236,0.001467,0.994178,0.001474,0.994177,0.001477,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993166,0.995227
67,pen-based,cosine,5,borda,RetentionPolicy.DD_RETENTION,10,9892.8,1099.2,0.001763,0.000115,...,0.001441,0.994236,0.001467,0.994178,0.001474,0.994177,0.001477,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993166,0.995227
58,pen-based,cosine,5,borda,RetentionPolicy.DIFFERENT_CLASS_RETENTION,10,9892.8,1099.2,0.001661,0.000112,...,0.001441,0.994236,0.001467,0.994178,0.001474,0.994177,0.001477,"{""labels"": [""0"", ""1"", ""2"", ""3"", ""4"", ""5"", ""6"",...",0.993166,0.995227
